<a href="https://colab.research.google.com/github/CH12YLmeno/Final-Group-Code/blob/main/INFS2822.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sydney Airbnb Data Preparation and Validation

## Data

This section inspects the supplied Sydney Airbnb dataset, documents the data-quality checks, and prepares the data for the later analysis. Two datasets are created:

- `df_clean`: the cleaned dataset used for analyses that do not require a complete price value.
- `price_data`: the subset of `df_clean` containing observed prices, used only for price-dependent analyses.


### Data loading

The following cell uploads the supplied CSV in Google Colab and reads the uploaded bytes directly. This avoids relying on a hard-coded filename when Colab renames a duplicate upload. A local fallback is included so that the same notebook can also be checked outside Colab.


In [ ]:
import io
from pathlib import Path

import pandas as pd

try:
    from google.colab import files
except ImportError:
    local_candidates = [
        Path("Sydney Airbnb (1).csv"),
        Path("Sydney Airbnb.csv")
    ]
    data_path = next(
        (path for path in local_candidates if path.exists()),
        None
    )

    if data_path is None:
        raise FileNotFoundError(
            "Upload the Sydney Airbnb CSV or place it beside the notebook."
        )

    loaded_name = data_path.name
    df = pd.read_csv(data_path, low_memory=False)
else:
    uploaded = files.upload()

    if not uploaded:
        raise ValueError("No CSV file was uploaded.")

    loaded_name, uploaded_bytes = next(iter(uploaded.items()))
    df = pd.read_csv(
        io.BytesIO(uploaded_bytes),
        low_memory=False
    )

print("Loaded file:", loaded_name)
print("Dataset shape:", df.shape)
display(df.head())


### Initial inspection

The structure, column names, data types, missing values, and duplicate records are reviewed before any values are changed.


In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("\nColumns:")
print(df.columns.tolist())
print("\nData types and non-missing counts:")
df.info()


In [ ]:
missing_summary = pd.DataFrame({
    "Missing values": df.isna().sum(),
    "Missing percentage": (df.isna().mean() * 100).round(2)
})

missing_summary = (
    missing_summary[missing_summary["Missing values"] > 0]
    .sort_values("Missing values", ascending=False)
)

display(missing_summary)


The dataset contains 20,833 listings and 19 variables. Missingness is concentrated in six fields. `neighbourhood_group` is entirely missing. There are 3,210 missing values in both `last_review` and `reviews_per_month`, 3,112 in `price`, 1,230 in `license`, and 13 in `minimum_nights`. The treatment of each field is documented below.


In [ ]:
duplicate_summary = pd.DataFrame({
    "Check": [
        "Duplicated complete rows",
        "Duplicated listing IDs",
        "Missing listing IDs",
        "Missing host IDs"
    ],
    "Rows": [
        int(df.duplicated().sum()),
        int(df["id"].duplicated().sum()),
        int(df["id"].isna().sum()),
        int(df["host_id"].isna().sum())
    ]
})

display(duplicate_summary)


No duplicated rows, duplicated listing identifiers, missing listing identifiers, or missing host identifiers were found. Each row can therefore be treated as a distinct listing record.


### Data-quality validation

The following checks test categorical consistency and basic field validity. A result of zero indicates that the tested issue was not found; it does not mean the data was assumed to be valid without inspection.


In [ ]:
categorical_columns = ["neighbourhood", "room_type"]
categorical_checks = []

for column in categorical_columns:
    raw_values = df[column].dropna().astype(str)
    stripped_values = raw_values.str.strip()
    normalised_values = stripped_values.str.casefold()

    raw_forms_per_normalised_label = (
        pd.DataFrame({
            "Raw value": raw_values,
            "Normalised value": normalised_values
        })
        .drop_duplicates()
        .groupby("Normalised value")["Raw value"]
        .nunique()
    )

    categorical_checks.append({
        "Column": column,
        "Unique labels": int(raw_values.nunique()),
        "Unique labels after normalisation": int(
            normalised_values.nunique()
        ),
        "Rows with surrounding whitespace": int(
            (raw_values != stripped_values).sum()
        ),
        "Case or whitespace conflict groups": int(
            (raw_forms_per_normalised_label > 1).sum()
        )
    })

categorical_check_summary = pd.DataFrame(categorical_checks)
display(categorical_check_summary)

print("Room-type labels and counts:")
display(df["room_type"].value_counts().rename("Listings"))


The 38 `neighbourhood` labels and four `room_type` labels remain unchanged after case-folding and whitespace removal. No surrounding whitespace or alternative case variants were detected, so no categorical labels required correction.


In [ ]:
minimum_nights_non_integer = (
    df["minimum_nights"].notna()
    & (df["minimum_nights"] % 1 != 0)
)

validity_conditions = {
    "Non-positive listing IDs": df["id"] <= 0,
    "Non-positive host IDs": df["host_id"] <= 0,
    "Non-positive observed prices": (
        df["price"].notna() & (df["price"] <= 0)
    ),
    "Minimum-night values below 1": (
        df["minimum_nights"].notna()
        & (df["minimum_nights"] < 1)
    ),
    "Non-integer observed minimum nights": minimum_nights_non_integer,
    "Negative total review counts": df["number_of_reviews"] < 0,
    "Negative reviews per month": (
        df["reviews_per_month"].notna()
        & (df["reviews_per_month"] < 0)
    ),
    "Host listing counts below 1": (
        df["calculated_host_listings_count"] < 1
    ),
    "Availability outside 0 to 365": (
        ~df["availability_365"].between(0, 365)
    ),
    "Negative reviews in the last 12 months": (
        df["number_of_reviews_ltm"] < 0
    ),
    "Last-12-month reviews above total reviews": (
        df["number_of_reviews_ltm"] > df["number_of_reviews"]
    ),
    "Latitude outside -90 to 90": (
        ~df["latitude"].between(-90, 90)
    ),
    "Longitude outside -180 to 180": (
        ~df["longitude"].between(-180, 180)
    )
}

validity_summary = pd.DataFrame({
    "Check": validity_conditions.keys(),
    "Invalid rows": [
        int(condition.sum())
        for condition in validity_conditions.values()
    ]
})

display(validity_summary)

range_columns = [
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365",
    "number_of_reviews_ltm"
]

display(
    df[range_columns]
    .agg(["min", "max"])
    .T
    .rename(columns={"min": "Minimum", "max": "Maximum"})
)


All validity checks returned zero invalid rows. In particular, review counts are non-negative, host listing counts are at least one, and availability falls within 0 to 365 days. The observed price maximum and 500-night maximum stay are potential outliers for later analytical consideration, but they are not impossible values and were not automatically removed.


In [ ]:
host_count_validation = (
    df.groupby("host_id")
    .agg(
        observed_listing_count=("id", "nunique"),
        reported_count_min=(
            "calculated_host_listings_count",
            "min"
        ),
        reported_count_max=(
            "calculated_host_listings_count",
            "max"
        )
    )
)

host_count_validation["reported_count_consistent"] = (
    host_count_validation["reported_count_min"]
    == host_count_validation["reported_count_max"]
)

host_count_validation["matches_observed_count"] = (
    host_count_validation["reported_count_consistent"]
    & (
        host_count_validation["reported_count_min"]
        == host_count_validation["observed_listing_count"]
    )
)

host_validation_summary = pd.DataFrame({
    "Metric": [
        "Hosts checked",
        "Hosts with inconsistent reported counts",
        "Hosts not matching unique listing IDs"
    ],
    "Count": [
        len(host_count_validation),
        int((~host_count_validation[
            "reported_count_consistent"
        ]).sum()),
        int((~host_count_validation[
            "matches_observed_count"
        ]).sum())
    ]
})

display(host_validation_summary)


All 9,769 hosts have one consistent `calculated_host_listings_count`, and every reported value equals the number of unique listing IDs associated with that host in the complete imported dataset. The field is therefore suitable for later classification of single-listing and multi-listing hosts. This validation is performed before row exclusions because the reported field describes the original supplied dataset.


### Cleaning decisions

The following steps create a separate working copy so that the original imported data remains unchanged.


In [ ]:
df_clean = df.copy()

df_clean = df_clean.drop(
    columns=["neighbourhood_group"]
)

print("Original shape:", df.shape)
print("Shape after removing neighbourhood_group:", df_clean.shape)


#### Removal of `neighbourhood_group`

The `neighbourhood_group` column was removed because all 20,833 entries were missing. Retaining a completely empty column would not contribute information to the analysis.


In [ ]:
non_missing_last_review_before = df_clean["last_review"].notna().sum()

df_clean["last_review"] = pd.to_datetime(
    df_clean["last_review"],
    errors="coerce"
)

non_missing_last_review_after = df_clean["last_review"].notna().sum()

print("Non-missing dates before conversion:", non_missing_last_review_before)
print("Valid dates after conversion:", non_missing_last_review_after)
print("Earliest review date:", df_clean["last_review"].min())
print("Latest review date:", df_clean["last_review"].max())


#### Review-date conversion

`last_review` was converted to a datetime type so that dates can be analysed correctly. All non-missing values converted successfully. The 3,210 missing dates remain missing because a review date cannot be inferred for a listing with no recorded review.


In [ ]:
zero_review_condition = (
    (df_clean["number_of_reviews"] == 0)
    & df_clean["reviews_per_month"].isna()
)

positive_reviews_missing_monthly_rate = (
    (df_clean["number_of_reviews"] > 0)
    & df_clean["reviews_per_month"].isna()
)

print(
    "Zero-review listings with missing reviews_per_month:",
    int(zero_review_condition.sum())
)
print(
    "Positive-review listings with missing reviews_per_month:",
    int(positive_reviews_missing_monthly_rate.sum())
)

df_clean.loc[
    zero_review_condition,
    "reviews_per_month"
] = 0

print(
    "Remaining missing reviews_per_month:",
    int(df_clean["reviews_per_month"].isna().sum())
)


#### Treatment of `reviews_per_month`

All 3,210 missing `reviews_per_month` values belong to listings with zero total reviews, while no listing with positive reviews has a missing monthly rate. These values were set to zero because zero is logically implied by the recorded review count; no statistical estimate was introduced.


In [ ]:
missing_license = int(
    df_clean["license"].isna().sum()
)
missing_license_percentage = (
    missing_license / len(df_clean) * 100
)

print("Rows with missing license information:", missing_license)
print(
    "Percentage of rows:",
    round(missing_license_percentage, 2),
    "%"
)
print("Rows removed because of missing license information: 0")


#### Treatment of missing licence information

The `license` field is not required for the planned analysis. Its 1,230 missing values, representing approximately 5.90% of the original dataset, were therefore retained as missing. Removing these listings solely because licence information is unavailable would unnecessarily reduce the sample without improving the variables used in the analysis. The original `license` column remains in the dataset for completeness and transparency.


In [ ]:
missing_minimum_nights = int(
    df_clean["minimum_nights"].isna().sum()
)
missing_minimum_nights_percentage = (
    missing_minimum_nights / len(df_clean) * 100
)

print("Rows with missing minimum_nights:", missing_minimum_nights)
print(
    "Percentage of rows:",
    round(missing_minimum_nights_percentage, 2),
    "%"
)

df_clean = df_clean.dropna(
    subset=["minimum_nights"]
).copy()

df_clean["minimum_nights"] = (
    df_clean["minimum_nights"].astype(int)
)

print("Rows remaining:", len(df_clean))


#### Removal of missing `minimum_nights` records

Thirteen listings, representing approximately 0.06% of the original dataset, had no `minimum_nights` value. These records were removed because the affected proportion is negligible and deletion avoids creating artificial minimum-stay requirements. The cleaned dataset retains 20,820 listings.


In [ ]:
price_data = df_clean.dropna(
    subset=["price"]
).copy()

print("Rows in df_clean:", len(df_clean))
print("Rows available for price analysis:", len(price_data))
print(
    "Rows excluded only from price analysis:",
    int(df_clean["price"].isna().sum())
)
print(
    "Missing-price percentage in df_clean:",
    round(df_clean["price"].isna().mean() * 100, 2),
    "%"
)


#### Treatment of missing prices

The original dataset contains 3,112 missing prices, representing approximately 14.94% of all listings. These values were retained as missing because replacing them would introduce estimated prices into the dataset, while deleting every affected listing would unnecessarily reduce the sample for analyses that do not use price. One of the 13 records removed for missing `minimum_nights` also had a missing price, leaving 3,111 missing prices in `df_clean`. Listings with missing prices are retained in `df_clean` and excluded only from price-dependent analysis through `price_data`, which contains 17,709 listings with observed prices.


### Final preparation audit

The final checks confirm the dataset dimensions, remaining missing values, data types, and descriptive ranges after the documented cleaning decisions.


In [ ]:
final_missing_summary = pd.DataFrame({
    "Missing values": df_clean.isna().sum(),
    "Missing percentage": (
        df_clean.isna().mean() * 100
    ).round(2)
})

final_missing_summary = (
    final_missing_summary[
        final_missing_summary["Missing values"] > 0
    ]
    .sort_values("Missing values", ascending=False)
)

print("Final df_clean shape:", df_clean.shape)
print("Final price_data shape:", price_data.shape)
display(final_missing_summary)

print("Final data types and non-missing counts:")
df_clean.info()

analysis_columns = [
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365",
    "number_of_reviews_ltm"
]

display(df_clean[analysis_columns].describe().T)


The final `df_clean` dataset contains 20,820 rows and 18 columns. Missing prices, review dates, and licence information are retained because they cannot be determined directly from the available information. The licence field is not used in the planned analysis. `price_data` contains the 17,709 records available for price-dependent analysis. The validated `calculated_host_listings_count` remains the appropriate field for classifying single-listing and multi-listing hosts.
